# TP5 - Predicción de suscripción a depósito a plazo (dataset bancario)
Entrenamiento de modelos de Machine Learning: Regresión Logística vs Árbol de Decisión (entropía).

Dataset: `dataset_banco.csv` (dataset original, sin preprocesar).

## 1. Importamos las librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import joblib

## 2. Cargamos los datos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
URL = "/content/drive/MyDrive/TP_Banco/dataset_banco.csv"

In [ ]:
train_df = pd.read_csv(URL, low_memory=False)

## 3. Análisis exploratorio de datos (EDA)

### 3.1 Primeras filas

In [ ]:
train_df.head(10)

### 3.2 Forma del dataset

In [ ]:
print("Rows x Columns:", train_df.shape)

### 3.3 Tipos de datos

In [ ]:
train_df.dtypes

### 3.4 Resumen estadístico (variables numéricas)

In [ ]:
train_df.describe()

### 3.5 Valores nulos por columna

In [ ]:
train_df.isna().sum()

### 3.6 Valores únicos en columnas categóricas (para detectar inconsistencias)

In [ ]:
cat_cols = ['job','marital','education','default','housing','loan','contact','month','poutcome','y']
for col in cat_cols:
    print(col, '->', sorted(train_df[col].dropna().astype(str).unique().tolist()))
    print()

**Hallazgos del EDA:**
- Hay valores nulos en casi todas las columnas.
- Hay inconsistencias de texto: mayúsculas/minúsculas mezcladas (`management`, `Management`, `MANAGEMENT`), abreviaturas (`div.` = `divorced`, `sec.` = `secondary`, `UNK` = `unknown`).
- Hay valores inválidos: `day` con un valor fuera de rango (`13131231221`), `month` con un valor numérico inválido (`156`), `contact` con un código sin sentido (`16`), `loan` con un valor que no corresponde a esa columna (`cellular`).
- Hay filas duplicadas.
- La variable objetivo `y` también tiene 1 valor nulo.

### 3.7 Duplicados

In [ ]:
print('Filas duplicadas:', train_df.duplicated().sum())

### 3.8 Balance de clases de la variable objetivo

In [ ]:
print(train_df['y'].value_counts(dropna=False))
print(train_df['y'].value_counts(normalize=True, dropna=False))
sns.countplot(x='y', data=train_df)
plt.title('Distribución de la variable objetivo (y)')
plt.show()

**Conclusión:** el dataset está desbalanceado (~88% 'no' / ~12% 'yes'). Se tendrá en cuenta al entrenar (`class_weight='balanced'`) y al elegir métricas de evaluación.

## 4. Limpieza y preprocesamiento de datos

### 4.1 Normalización de texto en columnas categóricas
Pasamos todo a minúsculas y sacamos espacios extra, para que valores como `Management` y `management` se traten como la misma categoría.

In [ ]:
cat_text_cols = ['job','marital','education','default','housing','loan','contact','month','poutcome','y']
for col in cat_text_cols:
    train_df[col] = train_df[col].astype(str).str.strip().str.lower()
    train_df[col] = train_df[col].replace('nan', np.nan)

### 4.2 Unificación de sinónimos / abreviaturas

In [ ]:
train_df['marital'] = train_df['marital'].replace({'div.': 'divorced'})
train_df['education'] = train_df['education'].replace({'sec.': 'secondary', 'unk': 'unknown'})
train_df['poutcome'] = train_df['poutcome'].replace({'unk': 'unknown'})
train_df['job'] = train_df['job'].replace({'administrative': 'admin.'})

### 4.3 Corrección de valores inválidos (se marcan como nulos para luego imputarlos)

In [ ]:
# day: debe estar entre 1 y 31
train_df['day'] = pd.to_numeric(train_df['day'], errors='coerce')
train_df.loc[(train_df['day'] < 1) | (train_df['day'] > 31), 'day'] = np.nan

# month: solo deben existir los 12 nombres de mes abreviados
meses_validos = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
train_df.loc[~train_df['month'].isin(meses_validos), 'month'] = np.nan

# contact: solo deben existir estas 3 categorías
train_df.loc[~train_df['contact'].isin(['cellular','telephone','unknown']), 'contact'] = np.nan

# loan: solo puede ser yes/no
train_df.loc[~train_df['loan'].isin(['yes','no']), 'loan'] = np.nan

### 4.4 Eliminación de duplicados y de filas sin variable objetivo

In [ ]:
train_df = train_df.drop_duplicates()
train_df = train_df.dropna(subset=['y'])
print('Shape luego de limpiar duplicados y target nulo:', train_df.shape)

### 4.5 Tratamiento de valores nulos (imputación)
- Variables numéricas: se imputan con la **mediana** (más robusta ante outliers que el promedio).
- Variables categóricas: se imputan con la **moda** (valor más frecuente).

In [ ]:
num_cols = ['age','balance','day','duration','campaign','pdays','previous']
for col in num_cols:
    train_df[col] = pd.to_numeric(train_df[col], errors='coerce')
    train_df[col] = train_df[col].fillna(train_df[col].median())

cat_cols_impute = ['job','marital','education','default','housing','loan','contact','month','poutcome']
for col in cat_cols_impute:
    train_df[col] = train_df[col].fillna(train_df[col].mode()[0])

print('Nulos restantes:', train_df.isna().sum().sum())

### 4.6 Codificación de la variable objetivo

In [ ]:
train_df['y'] = train_df['y'].map({'no': 0, 'yes': 1})

### 4.7 Matriz de correlación (variables numéricas)

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(train_df[num_cols + ['y']].corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de correlación')
plt.show()

## 5. Separamos variables de entrada (X) y variable objetivo (y)

In [ ]:
numeric_features = ['age','balance','day','duration','campaign','pdays','previous']
categorical_features = ['job','marital','education','default','housing','loan','contact','month','poutcome']

X = train_df[numeric_features + categorical_features]
y = train_df['y']

## 6. División en entrenamiento y evaluación
`stratify=y` mantiene la misma proporción 88/12 en ambos conjuntos.

In [ ]:
X_train, X_eval, y_train, y_eval = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train:', X_train.shape, 'Eval:', X_eval.shape)

## 7. Pipeline de preprocesamiento + modelo
Usamos un `ColumnTransformer` que:
- Estandariza (`StandardScaler`) las variables numéricas.
- Codifica (`OneHotEncoder`) las variables categóricas.

Esto se encapsula en un `Pipeline` junto con el modelo, para que **todo el preprocesamiento quede guardado dentro del mismo objeto entrenado** (ideal para después usarlo en Streamlit con datos nuevos sin procesar).

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

## 8. Modelo 1: Regresión Logística
**Justificación:** problema de clasificación binaria, modelo interpretable y eficiente, estándar como línea base.

In [ ]:
log_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
log_pipeline.fit(X_train, y_train)
y_pred_log = log_pipeline.predict(X_eval)

### 8.1 Métricas - Regresión Logística

In [ ]:
print('Accuracy :', accuracy_score(y_eval, y_pred_log))
print('Precision:', precision_score(y_eval, y_pred_log))
print('Recall   :', recall_score(y_eval, y_pred_log))
print('F1-score :', f1_score(y_eval, y_pred_log))
print()
print(classification_report(y_eval, y_pred_log))

In [ ]:
cm_log = confusion_matrix(y_eval, y_pred_log)
sns.heatmap(cm_log, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de Confusión - Regresión Logística')
plt.xlabel('Predicho'); plt.ylabel('Real')
plt.show()

## 9. Modelo 2 (comparación): Árbol de Decisión con criterio Entropía
**Justificación:** modelo no lineal, captura interacciones entre variables; `criterion='entropy'` según lo pedido.

In [ ]:
tree_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', DecisionTreeClassifier(criterion='entropy', max_depth=6, class_weight='balanced', random_state=42))
])
tree_pipeline.fit(X_train, y_train)
y_pred_tree = tree_pipeline.predict(X_eval)

### 9.1 Métricas - Árbol de Decisión

In [ ]:
print('Accuracy :', accuracy_score(y_eval, y_pred_tree))
print('Precision:', precision_score(y_eval, y_pred_tree))
print('Recall   :', recall_score(y_eval, y_pred_tree))
print('F1-score :', f1_score(y_eval, y_pred_tree))
print()
print(classification_report(y_eval, y_pred_tree))

In [ ]:
cm_tree = confusion_matrix(y_eval, y_pred_tree)
sns.heatmap(cm_tree, annot=True, fmt='d', cmap='Greens')
plt.title('Matriz de Confusión - Árbol de Decisión')
plt.xlabel('Predicho'); plt.ylabel('Real')
plt.show()

### 9.2 Importancia de variables (Árbol de Decisión)

In [ ]:
ohe = tree_pipeline.named_steps['prep'].named_transformers_['cat']
nombres_cat = ohe.get_feature_names_out(categorical_features)
nombres_totales = numeric_features + list(nombres_cat)
importances = pd.Series(tree_pipeline.named_steps['model'].feature_importances_, index=nombres_totales).sort_values(ascending=False)
importances.head(15).plot(kind='bar', figsize=(10,4), title='Importancia de variables - Árbol de Decisión (top 15)')
plt.tight_layout()
plt.show()

## 10. Comparación final de modelos

In [ ]:
comparacion = pd.DataFrame({
    'Modelo': ['Regresión Logística', 'Árbol de Decisión (entropía)'],
    'Accuracy': [accuracy_score(y_eval, y_pred_log), accuracy_score(y_eval, y_pred_tree)],
    'Precision': [precision_score(y_eval, y_pred_log), precision_score(y_eval, y_pred_tree)],
    'Recall': [recall_score(y_eval, y_pred_log), recall_score(y_eval, y_pred_tree)],
    'F1-score': [f1_score(y_eval, y_pred_log), f1_score(y_eval, y_pred_tree)],
})
comparacion

**Conclusión:** el Árbol de Decisión obtiene mejores métricas en todos los indicadores (especialmente F1-score), por lo que se elige como modelo final.

## 11. Guardamos el pipeline completo (preprocesamiento + modelo) para usar en Streamlit

In [ ]:
modelo_final = tree_pipeline  # pipeline completo: preprocesamiento + modelo
joblib.dump(modelo_final, 'modelo_banco.pkl')
print('Modelo guardado como modelo_banco.pkl')